# MACD: wrapper calls vs memoryviews under nogil

Compare two MACD compositions using the **same EMA loop**, three EMA passes, and five intermediate/output arrays:

- `calc_macd_wrapped` calls `calc_ema` three times and uses NumPy subtraction, like the production implementation.
- `calc_macd_nogil` converts the input once, allocates all arrays upfront, and calls `np_ema` plus subtraction loops inside one `with nogil` block.

A third control, `calc_macd_split_gil`, uses exactly the same allocations, memoryviews, and subtraction loops as `calc_macd_nogil`, but releases/reacquires the GIL around each of the five passes. This helps separate GIL transitions from the other changes. None of these variants fuses the EMA passes.

This experiment supports the unadjusted EMA used by MACD. Production code is unchanged. Timings include warmup, alternating measurement order, and numerical parity checks; compilation and dataset construction are excluded. The Polars cases use identical struct wrappers, including the same NaN-to-null conversion.

Run all cells in the mintalib environment. To compare Polars worker counts, start a fresh kernel with `POLARS_MAX_THREADS=1` or `POLARS_MAX_THREADS=11`; Polars initializes its pool once per process.

Allocation experiment: `calc_macd_packed` stores the three returned vectors in one C-contiguous `(3, size)` allocation and returns row views. The two temporary EMA arrays remain separate. It keeps the single-block variant's numeric passes and initialization (including filling the signal buffer with NaNs) unchanged, reducing five data allocations to three while adding row-view construction.

In [1]:
import os
import platform
import timeit
from statistics import median

from importlib.metadata import version
import numpy as np
import pandas as pd
import polars as pl
from polars.testing import assert_frame_equal

from mintalib import core
from mintalib.samples import sample_prices

%load_ext cython

print(f"Python {platform.python_version()}, Cython {version('Cython')}, NumPy {np.__version__}, Polars {pl.__version__}")
print(f"Polars workers: {pl.thread_pool_size()}; CPUs: {os.cpu_count()}")
print(f"Production core: {core.__file__}")

Python 3.11.14, Cython 3.2.4, NumPy 2.4.4, Polars 1.43.2
Polars workers: 11; CPUs: 12
Production core: /home/frederic/Projects/mintalib/src/mintalib/core.abi3.so


In [2]:
%%cython -c=-O3
# cython: boundscheck=False, wraparound=False, cdivision=True, nonecheck=False

import numpy as np
from collections import namedtuple
from libc.math cimport isnan, NAN

macd_result = namedtuple("macd_result", "macd macdsignal macdhist")

cdef void np_ema(const double[:] src, double[:] out, long period) noexcept nogil:
    # Caller validates period and matching buffers. Every output is written.
    cdef Py_ssize_t i, count = 0
    cdef double value, ema = NAN
    cdef double alpha = 2.0 / (period + 1.0)
    for i in range(src.shape[0]):
        value = src[i]
        out[i] = NAN
        if period == 1:
            out[i] = value
        elif not isnan(value):
            count += 1
            if isnan(ema):
                ema = value
            else:
                ema += alpha * (value - ema)
            if count >= period:
                out[i] = ema

cdef void np_subtract(const double[:] left, const double[:] right, double[:] out) noexcept nogil:
    cdef Py_ssize_t i
    for i in range(left.shape[0]):
        out[i] = left[i] - right[i]

def calc_ema(series, long period):
    if period <= 0:
        raise ValueError("period must be greater than zero")
    cdef const double[:] src = np.asarray(series, dtype=np.float64)
    result = np.full(src.shape[0], np.nan)
    cdef double[:] out = result
    with nogil:
        np_ema(src, out, period)
    return result

def calc_macd_wrapped(series, long n1=12, long n2=26, long n3=9):
    fast = calc_ema(series, n1)
    slow = calc_ema(series, n2)
    macd = fast - slow
    signal = calc_ema(macd, n3)
    hist = macd - signal
    return macd_result(macd, signal, hist)

def calc_macd_nogil(series, long n1=12, long n2=26, long n3=9):
    if n1 <= 0 or n2 <= 0 or n3 <= 0:
        raise ValueError("periods must be greater than zero")
    cdef const double[:] src = np.asarray(series, dtype=np.float64)
    cdef Py_ssize_t size = src.shape[0]
    fast_array = np.full(size, np.nan)
    slow_array = np.full(size, np.nan)
    macd_array = np.empty(size)
    signal_array = np.full(size, np.nan)
    hist_array = np.empty(size)
    cdef double[:] fast = fast_array, slow = slow_array
    cdef double[:] macd = macd_array, signal = signal_array, hist = hist_array
    with nogil:
        np_ema(src, fast, n1)
        np_ema(src, slow, n2)
        np_subtract(fast, slow, macd)
        np_ema(macd, signal, n3)
        np_subtract(macd, signal, hist)
    return macd_result(macd_array, signal_array, hist_array)

def calc_macd_split_gil(series, long n1=12, long n2=26, long n3=9):
    if n1 <= 0 or n2 <= 0 or n3 <= 0:
        raise ValueError("periods must be greater than zero")
    cdef const double[:] src = np.asarray(series, dtype=np.float64)
    cdef Py_ssize_t size = src.shape[0]
    fast_array = np.full(size, np.nan)
    slow_array = np.full(size, np.nan)
    macd_array = np.empty(size)
    signal_array = np.full(size, np.nan)
    hist_array = np.empty(size)
    cdef double[:] fast = fast_array, slow = slow_array
    cdef double[:] macd = macd_array, signal = signal_array, hist = hist_array
    with nogil:
        np_ema(src, fast, n1)
    with nogil:
        np_ema(src, slow, n2)
    with nogil:
        np_subtract(fast, slow, macd)
    with nogil:
        np_ema(macd, signal, n3)
    with nogil:
        np_subtract(macd, signal, hist)
    return macd_result(macd_array, signal_array, hist_array)

def calc_macd_packed(series, long n1=12, long n2=26, long n3=9):
    if n1 <= 0 or n2 <= 0 or n3 <= 0:
        raise ValueError("periods must be greater than zero")
    cdef const double[:] src = np.asarray(series, dtype=np.float64)
    cdef Py_ssize_t size = src.shape[0]
    fast_array = np.full(size, np.nan)
    slow_array = np.full(size, np.nan)
    # One allocation, three contiguous row views; no output copies.
    outputs = np.empty((3, size), dtype=np.float64)
    macd_array = outputs[0]
    signal_array = outputs[1]
    hist_array = outputs[2]
    # Match the baseline's signal initialization to isolate allocation layout.
    signal_array.fill(np.nan)
    cdef double[:] fast = fast_array, slow = slow_array
    cdef double[:] macd = macd_array, signal = signal_array, hist = hist_array
    with nogil:
        np_ema(src, fast, n1)
        np_ema(src, slow, n2)
        np_subtract(fast, slow, macd)
        np_ema(macd, signal, n3)
        np_subtract(macd, signal, hist)
    return macd_result(macd_array, signal_array, hist_array)


## Check numerical behavior

Check all output fields against production MACD, including warmup, missing values, empty/short inputs, noncontiguous input, and one-period EMAs. These checks also protect against accidentally changing mintalib's handling of NaNs while extracting the loop.

In [3]:
prices = pl.from_pandas(sample_prices().reset_index())
close = prices["close"].to_numpy()
kernels = {
    "production": core.calc_macd,
    "wrapped": calc_macd_wrapped,  # noqa: F821 — defined in %%cython
    "packed outputs": calc_macd_packed,  # noqa: F821 — defined in %%cython
    "one nogil block": calc_macd_nogil,  # noqa: F821 — defined in %%cython
    "split nogil blocks": calc_macd_split_gil,  # noqa: F821 — defined in %%cython
}
rng = np.random.default_rng(42)
walk = 100 + rng.normal(size=250).cumsum()
gaps = walk.copy()
gaps[[0, 1, 30, 80, 81, 170]] = np.nan
inputs = [close, walk, gaps, walk[::2], np.array([]), walk[:5], np.full(50, np.nan)]
parameters = [(12, 26, 9), (1, 1, 1), (1, 26, 9), (12, 26, 1), (3, 5, 2)]
for values in inputs:
    for period in [1, 2, 12, 26]:
        np.testing.assert_allclose(calc_ema(values, period), core.calc_ema(values, period), rtol=1e-12, atol=1e-12)  # noqa: F821 — defined in %%cython
    for periods in parameters:
        expected = core.calc_macd(values, *periods)
        for kernel in kernels.values():
            actual = kernel(values, *periods)
            for left, right in zip(actual, expected):
                np.testing.assert_allclose(left, right, rtol=1e-12, atol=1e-12)
print("EMA and all three MACD fields match production for every case.")
print(f"Single ticker: {len(prices):,} bars")
packed = calc_macd_packed(close)  # noqa: F821 — defined in %%cython
assert packed.macd.base is packed.macdsignal.base is packed.macdhist.base
assert packed.macd.base.shape == (3, len(close))
assert all(array.flags.c_contiguous for array in packed)
print("Packed outputs are contiguous views of one shared allocation.")

EMA and all three MACD fields match production for every case.
Single ticker: 11,504 bars
Packed outputs are contiguous views of one shared allocation.


In [4]:
def benchmark(cases, *, repeat=11, number=1):
    for call in cases.values():
        call()
    timings = {name: [] for name in cases}
    for iteration in range(repeat):
        names = list(cases) if iteration % 2 == 0 else list(reversed(cases))
        for name in names:
            elapsed = timeit.timeit(cases[name], number=number) / number
            timings[name].append(elapsed * 1000)
    return pd.DataFrame([
        {"variant": name, "min_ms": min(values), "median_ms": median(values)}
        for name, values in timings.items()
    ]).set_index("variant")

array_results = benchmark({name: lambda kernel=kernel: kernel(close, 12, 26, 9) for name, kernel in kernels.items()}, number=100)
array_results

,min_ms,median_ms
variant,,
production,0.084974,0.087079
wrapped,0.084366,0.086530
packed outputs,0.086406,0.088993
one nogil block,0.086739,0.088410
split nogil blocks,0.086895,0.088232


## Identical Polars wrappers

Each kernel is called once per ticker and returns one struct with all three fields. The dataset repeats the bundled daily prices for 500 synthetic tickers, sorted by ticker and existing row order. Grouping, execution, output construction, and result disposal are included in timings.

In [5]:
FIELDS = ("macd", "macdsignal", "macdhist")
DTYPE = pl.Struct({name: pl.Float64 for name in FIELDS})

def macd_expr(kernel):
    def batch(series):
        result = kernel(series, 12, 26, 9)
        return pl.DataFrame(result, schema=FIELDS, orient="col", nan_to_null=True).to_struct()
    return pl.col("close").map_batches(batch, return_dtype=DTYPE).alias("result")

expressions = {name: macd_expr(kernel) for name, kernel in kernels.items()}
dataset = pl.concat([
    prices.select(pl.lit(f"T{i:03d}").alias("ticker"), pl.col("close"))
    for i in range(1, 501)
]).rechunk()
print(f"Grouped workload: {dataset.height:,} rows, {dataset['ticker'].n_unique()} tickers")
polars_results = {}
for label, data, grouped in [("single ticker", prices, False), ("500 tickers", dataset, True)]:
    selected = {name: expr.over("ticker") if grouped else expr for name, expr in expressions.items()}
    reference = data.select(selected["production"]).unnest("result")
    for expr in selected.values():
        assert_frame_equal(data.select(expr).unnest("result"), reference, rel_tol=1e-12, abs_tol=1e-12)
    polars_results[label] = benchmark(
        {name: lambda expr=expr: data.select(expr) for name, expr in selected.items()},
        number=1 if grouped else 20,
    )
print("All Polars fields and nulls match production.")
pd.concat(polars_results, names=["workload"])

Grouped workload: 5,752,000 rows, 500 tickers


All Polars fields and nulls match production.


min_ms   median_ms
workload      variant                                   
single ticker production            0.116869    0.120106
              wrapped               0.116906    0.121700
              packed outputs        0.117304    0.120550
              one nogil block       0.117569    0.121183
              split nogil blocks    0.118052    0.121360
500 tickers   production          111.062091  122.157996
              wrapped             117.603376  118.713291
              packed outputs       86.537117   88.820198
              one nogil block      83.993411   91.677903
              split nogil blocks  102.068183  110.712425

## Interpretation

Compare **one nogil block** with **split nogil blocks** first: those variants differ only in where they release/reacquire the GIL. Compare **wrapped** with **one nogil block** for the combined effect of avoiding wrapper calls, repeated conversion/setup, and NumPy subtraction dispatch as well as reducing GIL transitions. That second comparison does not isolate the GIL alone.

The production baseline is compiled separately and has a more general EMA implementation, so compare the notebook variants with each other for controlled conclusions. Repeat with a fresh one-worker kernel to see whether differences depend on contention. Absolute timings vary with CPU load and worker count; do not generalize a single run to all input lengths.

Compare **packed outputs** with **one nogil block** to assess allocating the three outputs together. Both keep identical passes, initialization, and GIL regions; the packed variant adds three NumPy row views and shares the allocation lifetime across the returned fields.

## Initial comparison before the packed-output experiment (2026-09-08)

Fresh kernel executions with 1 and 11 Polars workers, 11 timing rounds per case. This notebook uses mintalib's current bundled sample: **11,504 bars per ticker, 5,752,000 grouped rows**. The earlier barcalc comparison used a different bundled sample (11,006 bars), so compare variants within this experiment rather than comparing absolute times across notebooks/scripts. These are the initial measurements; execution outputs above may reflect a later run including the packed-output variant.

| MACD variant | 1 worker median (ms) | 11 workers median (ms) |
|---|---:|---:|
| Production | 80.78 | 121.19 |
| Wrapped | 80.17 | 117.24 |
| One nogil block | 79.19 | 91.90 |
| Split nogil blocks | 79.64 | 108.60 |

All numerical checks passed in both executions. Direct-array medians were approximately 0.086–0.091 ms, with no meaningful advantage for the single-block variant.

At 11 workers, changing only five GIL-release regions to one reduced the split-block control's median from 108.60 to 91.90 ms (**15% less time**). Relative to wrapper composition, the single-block variant used **22% less time**. With one worker, the three experimental variants were within about 1 ms of one another. This supports a contention-related cost from repeated GIL transitions; it does not assign the entire wrapper-composition difference to the GIL. Even the single-block version remained slower with 11 workers than with one on this workload.

## Packed-output allocation experiment (2026-09-08)

The three returned vectors now share one `np.empty((3, size))` allocation, with contiguous row views; the two EMA temporary buffers remain separate. Signal initialization, numeric passes, and the single `nogil` block are unchanged. This reduces data allocations from five to three, but still constructs the row-view objects under the GIL. The output fields share the backing allocation's lifetime.

500 tickers × 11,504 bars, 11 alternating timing rounds per fresh kernel:

| Output allocation | 1 worker median (ms) | 11 workers median (ms) | 11 workers minimum (ms) |
|---|---:|---:|---:|
| Separate output arrays | 80.37 | 91.68 | 83.99 |
| One packed output array | 80.26 | 88.82 | 86.54 |

The packed version's 11-worker median was about **3% lower**, while its minimum was worse. One-worker and direct-array timings were essentially unchanged. This run shows no convincing large improvement; the small median difference could be measurement variation. All numerical/Polars parity checks passed in both runs, and the notebook verifies that the returned fields are contiguous views of one shared allocation. Production kernels are unchanged.